# 04 — First model

Trains the first model and checks it honestly. Training uses **2014–2022**,
scoring uses **2023–2024** (validation). The test years, 2025 onwards, stay closed:
every number here is allowed to influence decisions, so none of it may come
from the test years.

**Men's fights are the headline**, because they are what this project is for.
The model still trains on all fights: tested on men's validation fights, the
pooled model beat one trained on men's fights only.

The logic lives in `src/ufc_predictor/modeling/` (`model.py`, `evaluate.py`,
`variants.py`). This notebook calls it and shows the results.

| Number | Meaning | Coin flip |
|---|---|---|
| accuracy | share of fights where the predicted favourite won | 0.500 |
| log loss | how surprised the predictions were by what happened; lower is better, confident mistakes cost most. **The main number.** | 0.693 |
| AUC | how well the predictions rank fights | 0.500 |

In [1]:
import numpy as np
import pandas as pd

from ufc_predictor.data.io import load_fighters, load_processed
from ufc_predictor.features import manifest
from ufc_predictor.features.elo import load_elo_params
from ufc_predictor.features.shrinkage import load_alphas
from ufc_predictor.modeling.evaluate import (
    calibration_table, compare, fit_baseline, group_importance,
    is_womens_fight, predict_baseline, scorecard,
)
from ufc_predictor.modeling.model import corner_disagreement, fit_model, mirror, predict_proba
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.variants import build_variants

featured = load_processed("fights_featured")
train, validation, _ = chronological_split(featured)      # the test years stay closed

X_train, y_train = manifest.make_xy(train)
X_val, y_val = manifest.make_xy(validation)
mens = ~is_womens_fight(validation, history=featured).to_numpy()

print(f"train:      {len(X_train):,} fights, {X_train.shape[1]} features")
print(f"validation: {len(X_val):,} fights, of which {mens.sum():,} men's")

train:      4,309 fights, 108 features
validation: 1,017 fights, of which 837 men's


## Baselines: what the model has to beat

Three simple predictors, each fitted on the training years only:

- **coin flip** — 50% for every fight.
- **age only** — a one-number rule from the age gap: younger fighters win more often.
- **win rate only** — the same, from the gap in (shrunk) win rate.

Both one-feature baselines have no built-in preference for either corner, like the model.

In [2]:
predictions = {"coin flip": np.full(len(y_val), 0.5)}
for column, name in [("diff_static_age", "age only"), ("diff_prior_win_rate", "win rate only")]:
    baseline = fit_baseline(X_train, y_train, column)
    predictions[name] = predict_baseline(baseline, X_val, column)

scorecard(predictions, y_val, mens)

,men_accuracy,men_log_loss,men_auc,all_accuracy,all_log_loss,all_auc
model,,,,,,
coin flip,0.5000,0.6931,0.5000,0.5000,0.6931,0.5000
age only,0.5926,0.6713,0.6208,0.5851,0.6716,0.6170
win rate only,0.5795,0.6754,0.6113,0.5762,0.6773,0.6051


## The first model

Gradient-boosted trees (scikit-learn's `HistGradientBoostingClassifier`) on all
the features (the count is printed above), with deliberately cautious, **untuned**
settings. Every fight is predicted twice, once with each fighter in corner A, and
the two answers are averaged (see the next section for why). The tuned settings,
and whether the Elo rating earns its place, are in `05_elo.ipynb`.

In [3]:
model = fit_model(X_train, y_train)
predictions["first model"] = predict_proba(model, X_val)

card = scorecard(predictions, y_val, mens)
card

,men_accuracy,men_log_loss,men_auc,all_accuracy,all_log_loss,all_auc
model,,,,,,
coin flip,0.5000,0.6931,0.5000,0.5000,0.6931,0.5000
age only,0.5926,0.6713,0.6208,0.5851,0.6716,0.6170
win rate only,0.5795,0.6754,0.6113,0.5762,0.6773,0.6051
first model,0.6141,0.6528,0.6642,0.6087,0.6564,0.6583


In [4]:
# is the gain over the best baseline more than luck? (men's fights, paired test)
best = card.drop(index="first model")["men_log_loss"].idxmin()
result = compare(y_val[mens], predictions["first model"][mens], predictions[best][mens])
print(f"first model vs '{best}' on men's fights: fixes {result['fixes']}, "
      f"breaks {result['breaks']}, p = {result['p_value']:.4f}")

first model vs 'age only' on men's fights: fixes 168, breaks 147, p = 0.2598


## Why every fight is predicted both ways round

Fighter A and fighter B are a coin flip, so the same fight can be shown to the
model two ways. Trees don't know that. Asked one way only, the model sometimes
picks a different winner depending on who happens to be in corner A. Averaging
the two answers removes the contradiction completely.

In [5]:
print(f"asked one way only, the model contradicts itself on {corner_disagreement(model, X_val):.1%} of fights")

swapped = predict_proba(model, mirror(X_val))
gap = np.abs(predictions["first model"] + swapped - 1).max()
print(f"averaged: p(A wins) + p(B wins) is 1 for every fight (largest gap {gap:.1e})")

asked one way only, the model contradicts itself on 26.8% of fights
averaged: p(A wins) + p(B wins) is 1 for every fight (largest gap 1.1e-16)


## Does shrinkage actually help the model?

Shrinkage was chosen because it predicts each fighter's *next fight* better.
Whether it also helps predict *who wins* is a separate question. The same model
is trained on three versions of the ratio features:

- **raw** — no shrinkage: accuracy is NaN when a fighter never attempted something.
- **zero_fill** — raw, with those 0/0 accuracies set to 0.
- **shrunk** — the pipeline's version.

The model handles NaN natively, so "raw" is a real contender.

In [6]:
variants = build_variants(load_processed("fights"), load_alphas(), load_fighters(), load_elo_params())

variant_predictions = {}
for name, table in variants.items():
    v_train, v_val, _ = chronological_split(table)
    assert (v_val["fights_url"].to_numpy() == validation["fights_url"].to_numpy()).all()
    Xt, yt = manifest.make_xy(v_train)
    Xv, _ = manifest.make_xy(v_val)
    variant_predictions[name] = predict_proba(fit_model(Xt, yt), Xv)

scorecard(variant_predictions, y_val, mens)

,men_accuracy,men_log_loss,men_auc,all_accuracy,all_log_loss,all_auc
model,,,,,,
raw,0.6033,0.6613,0.6473,0.6018,0.6626,0.6439
zero_fill,0.5938,0.6659,0.6383,0.5949,0.6668,0.6354
shrunk,0.6141,0.6528,0.6642,0.6087,0.6564,0.6583


In [7]:
for other in ("raw", "zero_fill"):
    result = compare(y_val[mens], variant_predictions["shrunk"][mens], variant_predictions[other][mens])
    print(f"shrunk vs {other:9s} on men's fights: fixes {result['fixes']:3d}, "
          f"breaks {result['breaks']:3d}, p = {result['p_value']:.3f}")

shrunk vs raw       on men's fights: fixes  81, breaks  72, p = 0.518
shrunk vs zero_fill on men's fights: fixes  82, breaks  65, p = 0.187


## What is the model relying on?

Each family of features (see `manifest.py`) is scrambled in turn on the men's
validation fights, and the rise in log loss is measured. A family the model
depends on hurts when scrambled; one it ignores barely moves. This uses fights
the model never trained on: on training fights the measurement would mostly
show what the model had memorised.

In [8]:
groups = manifest.feature_groups(train)
importance = group_importance(model, X_val[mens], y_val[mens], groups, repeats=20)
importance

,group,columns,log_loss_rise,spread
0,physical,7,0.033991,0.004405
1,strength,1,0.019576,0.004922
2,defence,44,0.014516,0.004992
3,striking,32,0.011553,0.004694
4,grappling,10,0.005977,0.003525
5,damage,2,0.000159,0.001406
6,cold_start,2,-0.000467,0.000592
7,record,8,-0.001490,0.001636
8,pace,2,-0.002710,0.001507


## Are the probabilities honest?

Men's validation fights are split into five equal groups by predicted
probability. If the model says 65% for a group, fighter A should win about 65%
of those fights.

In [9]:
calibration_table(y_val[mens], predictions["first model"][mens], bins=5)

,fights,predicted,actual
0,168,0.290,0.232
1,167,0.423,0.419
2,167,0.509,0.473
3,167,0.595,0.545
4,168,0.712,0.643
